# Regression: Predicting Age from Brain Gene Expression
### OLS, Ridge and LASSO

---

**Biological Background**

Gene expression in the brain changes with age. Genes involved in the immune and
stress responses are expressed more in older brains, while genes involved in
synapses and neurotransmission are expressed less. We use the dataset of
González-Velasco et al. (2020), which measures 1,077 brain-ageing genes (cortex region,
[doi:10.1016/j.bbagrm.2020.194491](https://doi.org/10.1016/j.bbagrm.2020.194491)) in
post-mortem human brain samples from 932 donors aged 2 to 102.

The task is to predict the chronological age of a donor (in years) from the expression
profile. This is a **regression** problem because the target is a continuous quantity
rather than a class label.

In [ ]:
# Standard scientific Python stack
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.base import clone
from sklearn.linear_model import (ElasticNet, ElasticNetCV, Lasso, LassoCV,
                                  lasso_path, LinearRegression, Ridge)
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot aesthetics
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("All imports successful!")
print(f"scikit-learn version: {__import__('sklearn').__version__}")

---
## Part 1: Loading the Data

The two CSV files store one gene per row (ENSEMBL identifier as index) and one sample
per column; the donor's age is recorded in a row named `"age"`. We transpose them into
the conventional *samples × features* layout and split off the target vector.

In [ ]:
# Load the course data (keep the CSVs next to this notebook)
train_csv = pd.read_csv("aging_training.csv", index_col=0)
test_csv = pd.read_csv("aging_testing.csv", index_col=0)

# Transpose: genes become columns (features), samples become rows.
# The "age" row becomes the target vector y.
X_train = train_csv.drop(index="age").T.astype(float)
y_train = train_csv.loc["age"].astype(float)

X_test = test_csv.drop(index="age").T.astype(float)
y_test = test_csv.loc["age"].astype(float)

# Shapes, age range, missing values, identical gene sets
print(f"Training set: {X_train.shape[0]} samples x {X_train.shape[1]} genes")
print(f"Test set:     {X_test.shape[0]} samples x {X_test.shape[1]} genes")
print(f"Age range: {y_train.min():.0f}-{y_train.max():.0f} (train), "
      f"{y_test.min():.0f}-{y_test.max():.0f} (test)")
print(f"Missing values: {X_train.isna().sum().sum() + X_test.isna().sum().sum()}")
print(f"Gene sets identical between splits: {set(X_train.columns) == set(X_test.columns)}")

# Discussion prompt:
# Q: The 70/30 split was provided with the data. Why is it important that we
#    never re-split and never use the test samples when making modelling choices?

---
## Part 2: Exploratory Analysis

We first look at the age distribution, the spread of the features, and the
correlation of individual genes with age.

In [ ]:
# Age distribution in both splits
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.arange(0, 105, 5)
ax.hist(y_train, bins=bins, alpha=0.7, label='train', color='steelblue')
ax.hist(y_test, bins=bins, alpha=0.7, label='test', color='tomato')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Number of donors')
ax.set_title('Age distribution of post-mortem brain samples')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-gene location and spread vary by orders of magnitude
mean_per_gene = X_train.mean(axis=0)
std_per_gene = X_train.std(axis=0)

print("Per-gene mean:")
print(mean_per_gene.describe().round(3))
print("\nPer-gene std:")
print(std_per_gene.describe().round(3))

# Discussion prompt:
# Q: Ridge and LASSO penalize every gene equally. What goes wrong if gene A
#    varies in the thousands and gene B in fractions?

In [ ]:
# Correlate every gene with age (training data only)
corr = X_train.corrwith(y_train)
top_gene = corr.abs().idxmax()

# Scatter the strongest gene against age
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_train[top_gene], y_train, s=14, alpha=0.6, color='steelblue')
ax.set_xlabel(f'{top_gene} expression')
ax.set_ylabel('Age (years)')
ax.set_title(f'Strongest gene-age correlation: {top_gene} (r = {corr[top_gene]:.2f})')
plt.tight_layout()
plt.show()

# The five strongest absolute correlations, for orientation
top5 = corr.abs().sort_values(ascending=False).head(5).index
print(corr.reindex(top5).round(2))

# Discussion prompt:
# Q: The strongest single-gene correlation is around |r| = 0.6. One gene is a
#    weak predictor. Where will the predictive power of the model come from?

---
## Part 3: Linear Regression Baseline

### How ordinary least squares works

Linear regression predicts age as a weighted sum of gene expressions,

$$\hat{y} = \beta_0 + \sum_{j=1}^{p} \beta_j x_j,$$

where the weights $\beta$ are chosen to minimise the **residual sum of squares**, the
total squared distance between predicted and true ages.

With 1,077 features but only 652 samples there are far more weights than data points.
The model can nearly memorise the training samples and therefore learns noise instead
of signal. This is called **over-fitting**. Whether the model **generalises** to new
donors is measured by **cross-validation (CV)**. The training data is split into 5
parts (**folds**). The model is fit on four folds and predicts the fifth, and this is
rotated until every fold was held out once. The five out-of-fold scores are averaged.

Three regression metrics, all computed out-of-fold:

* **MAE** (mean absolute error): the average prediction error, in years.
* **RMSE** (root mean squared error): also in years, but large errors count extra.
* **R²**: fraction of the variability in age explained; 1 = perfect, 0 = no better than
  always guessing the average age.

In [ ]:
# Cross-validated baseline: plain OLS on the training data.
# IMPORTANT: shuffle=True matters. The samples in the CSV carry order structure,
# and an unshuffled KFold produces one fold of almost only young donors.
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ols = LinearRegression()

cv_r2 = cross_val_score(ols, X_train, y_train, cv=kf, scoring='r2')
print(f"OLS 5-fold CV R2: {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}")

# The same model evaluated on the data it was fit on, for comparison
ols.fit(X_train, y_train)
train_r2 = ols.score(X_train, y_train)
print(f"OLS training R2:  {train_r2:.3f}")

# Discussion prompt:
# Q: The model fits the training samples almost perfectly but generalises
#    clearly worse. What does this gap measure, and why is CV the honest
#    estimate of the two?

---
## Part 4: Regularization: Ridge, LASSO, Elastic Net

### Why regularize?

With more weights than samples, infinitely many weight vectors fit the training data
equally well. Regularization picks among them by penalizing large weights. Simpler
models make more systematic errors (**bias**) but are less sensitive to the particular
training sample (**variance**). This is the **bias-variance trade-off**.

| method | penalty added to the loss | effect |
|---|---|---|
| Ridge | λ · Σ βⱼ²  (L2) | shrinks all weights towards zero |
| LASSO | λ · Σ ∣βⱼ∣  (L1) | shrinks *and selects*: some βⱼ become exactly 0 |
| Elastic Net | mix of both: (1−α) · L2 + α · L1 | compromise; keeps correlated genes together |

The penalty strength λ (called `alpha` in scikit-learn) is a **hyperparameter**, a
setting that is chosen rather than learned from data. It is tuned by cross-validation on the
training data only. Penalized models are scale-sensitive, so features are
**standardized** (rescaled to mean 0, variance 1) first; a scikit-learn `Pipeline`
applies the standardization inside every CV fold, so the held-out fold never
influences it.

### Ridge

We tune the penalty strength with a grid search over 25 alpha values.

In [ ]:
# Ridge penalty tuning by grid search over 25 alpha values.
# IMPORTANT: the scaler lives inside the Pipeline, so it is re-fit on the four
# training folds of every split; the held-out fold never influences it.
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge()),
])

ridge_alphas = np.logspace(-3, 5, 25)
ridge_grid = GridSearchCV(
    ridge_pipe,
    param_grid={'model__alpha': ridge_alphas},
    scoring='neg_root_mean_squared_error',
    cv=kf,
    n_jobs=-1,
)
ridge_grid.fit(X_train, y_train)

# CV RMSE against the penalty strength, on a log axis
cv_rmse_ridge = -ridge_grid.cv_results_['mean_test_score']
best_alpha_ridge = ridge_grid.best_params_['model__alpha']

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(ridge_alphas, cv_rmse_ridge, marker='o', markersize=4, color='steelblue')
ax.axvline(best_alpha_ridge, linestyle='--', color='tomato',
           label=f'best alpha = {best_alpha_ridge:.3g}')
ax.set_xlabel('alpha (penalty strength)')
ax.set_ylabel('5-fold CV RMSE (years)')
ax.set_title('Ridge: CV error vs penalty strength')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best alpha by 5-fold CV: {best_alpha_ridge:.4g} "
      f"(CV RMSE: {ridge_grid.best_score_ * -1:.2f} years)")

# Discussion prompt:
# Q: The curve is U-shaped. Why is the error large for small alpha and for
#    large alpha, and why is the middle best?

### LASSO

`LassoCV` computes the entire LASSO regularization path (the solution for a whole
range of penalties, reusing previous solutions as starting points) and cross-validates
along it.

In [ ]:
# LASSO path cross-validation on standardized features.
# Features are standardized once for the path search; the final refit for
# Part 5 wraps scaler and LASSO in a Pipeline again.
scaler = StandardScaler().fit(X_train)
Xs_train = scaler.transform(X_train)

lasso_cv = LassoCV(alphas=np.logspace(-2, 1, 16), cv=kf, tol=3e-3,
                   max_iter=100_000, n_jobs=-1)
lasso_cv.fit(Xs_train, y_train)

# CV RMSE against the penalty strength
cv_rmse_lasso = np.sqrt(lasso_cv.mse_path_.mean(axis=1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(lasso_cv.alphas_, cv_rmse_lasso, marker='o', markersize=4, color='steelblue')
ax.axvline(lasso_cv.alpha_, linestyle='--', color='tomato',
           label=f'best alpha = {lasso_cv.alpha_:.3g}')
ax.set_xlabel('alpha (penalty strength)')
ax.set_ylabel('5-fold CV RMSE (years)')
ax.set_title('LASSO: CV error vs penalty strength')
ax.legend()
plt.tight_layout()
plt.show()

# How many genes does the best LASSO model actually use?
n_selected = np.count_nonzero(lasso_cv.coef_)
print(f"Best alpha by 5-fold CV: {lasso_cv.alpha_:.4g} "
      f"(CV RMSE: {cv_rmse_lasso.min():.2f} years)")
print(f"LASSO uses {n_selected} of {X_train.shape[1]} genes")

# Refit the chosen penalty in a clean pipeline (scaler + LASSO) for Part 5
best_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=lasso_cv.alpha_, tol=3e-3, max_iter=100_000)),
]).fit(X_train, y_train)

# Discussion prompt:
# Q: LASSO keeps about 180 of 1,077 genes at essentially Ridge's accuracy.
#    What could make such a compact gene panel interesting beyond prediction?

### Coefficient paths

As α grows, weights are pulled towards zero. Ridge's L2 penalty only approaches zero
asymptotically, while LASSO's L1 penalty sets weights **exactly** to zero. This is why
LASSO performs feature selection.

In [ ]:
# Per-gene weight trajectories over the penalty strength for Ridge and LASSO
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Ridge path: refit Ridge for each alpha, keep all 1,077 coefficients
ridge_alphas_path = np.logspace(-1, 5, 30)
ridge_paths = np.array([Ridge(alpha=a).fit(Xs_train, y_train).coef_
                        for a in ridge_alphas_path])
for coefficients in ridge_paths.T:
    axes[0].plot(ridge_alphas_path, coefficients, linewidth=0.6, alpha=0.5)
axes[0].set_xscale('log')
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('coefficient value')
axes[0].set_title('Ridge (L2)')

# LASSO path: the path algorithm returns one coefficient vector per alpha
lasso_alphas_path, lasso_coefs, _ = lasso_path(
    Xs_train, y_train, alphas=np.logspace(-1, 2, 12)[::-1], tol=1e-3, max_iter=100_000)
for coefficients in lasso_coefs:
    axes[1].plot(lasso_alphas_path, coefficients, linewidth=0.6, alpha=0.5)
axes[1].set_xscale('log')
axes[1].set_xlabel('alpha')
axes[1].set_title('LASSO (L1)')

fig.suptitle('Coefficient paths: one line per gene')
plt.tight_layout()
plt.show()

# Discussion prompt:
# Q: Watch the two panels as alpha grows. Why do the Ridge lines approach zero
#    only asymptotically while the LASSO lines hit exactly zero?

### Elastic Net

Elastic Net mixes both penalties. With `l1_ratio` = 1 it is pure LASSO and with 0 it
is pure Ridge. Both hyperparameters are tuned jointly, and the chosen configuration is
refitted in a pipeline for the final comparison.

In [ ]:
# Joint tuning of penalty strength and L1/L2 mixture.
# The loose tolerance keeps the grid search fast; the same tolerance is used
# for the refit below.
enet_cv = ElasticNetCV(
    alphas=np.logspace(-2, 1, 8),
    l1_ratio=[0.5, 0.9, 0.95, 0.99],
    cv=kf,
    tol=1e-2,
    max_iter=100_000,
    n_jobs=-1,
)
enet_cv.fit(Xs_train, y_train)
print(f"Best alpha by 5-fold CV: {enet_cv.alpha_:.4g}, l1_ratio: {enet_cv.l1_ratio_}")

# Refit the chosen configuration in a clean pipeline for Part 5
best_enet = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_,
                         tol=1e-2, max_iter=100_000)),
]).fit(X_train, y_train)

---
## Part 5: Evaluation on the Test Set

### Why the test set is only used now

Everything so far happened **inside the training data**. The 280 test samples have not
influenced any decision so far, neither the exploration, nor the penalty grids, nor
the standardization. They are used **once**, now, to estimate real-world performance. If
any tuning step had used the test set, the resulting score would be optimistically
biased and would not replicate on truly new samples.

In [ ]:
# One evaluation protocol for every model:
#   1. mean out-of-fold metrics from 5-fold CV on the training data
#   2. a refit on the full training set and a single prediction of the test set
linear_models = {
    'OLS': LinearRegression(),
    'Ridge': ridge_grid.best_estimator_,
    'LASSO': best_lasso,
    'Elastic Net': best_enet,
}

results = {}
for name, model in linear_models.items():
    # CV metrics: the model is re-fit inside every fold by cross_val_score
    cv_mae = -cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
    cv_rmse = -cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
    cv_r2 = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')

    # Final fit on the full training set, single prediction of the test set
    fitted = clone(model).fit(X_train, y_train)
    pred = fitted.predict(X_test)
    residual = y_test - pred

    results[name] = {
        'CV MAE': cv_mae.mean(),
        'CV RMSE': cv_rmse.mean(),
        'CV R2': cv_r2.mean(),
        'test MAE': np.mean(np.abs(residual)),
        'test RMSE': np.sqrt(np.mean(residual ** 2)),
        'test R2': 1 - np.sum(residual ** 2) / np.sum((y_test - y_test.mean()) ** 2),
    }

results_df = pd.DataFrame(results).T
results_df.round(2)

In [ ]:
# Predicted vs true age and the residual pattern for the best model
best_name = results_df['test R2'].idxmax()
best_model = clone(linear_models[best_name]).fit(X_train, y_train)
pred = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, pred, s=14, alpha=0.6, color='steelblue')
axes[0].plot([0, 105], [0, 105], linestyle='--', color='tomato')
axes[0].set_xlabel('True age (years)')
axes[0].set_ylabel('Predicted age (years)')
axes[0].set_title(best_name)

axes[1].scatter(y_test, y_test - pred, s=14, alpha=0.6, color='steelblue')
axes[1].axhline(0, linestyle='--', color='tomato')
axes[1].set_xlabel('True age (years)')
axes[1].set_ylabel('Residual (years)')
axes[1].set_title(f'{best_name}: residuals')

plt.tight_layout()
plt.show()

# Discussion prompt:
# Q: Residuals scatter randomly around zero for most of the lifespan but drift
#    upwards for the oldest donors. What does that drift mean in plain words?

---
## Part 6: Summary and Discussion

| | **OLS** | **Ridge** | **LASSO** | **Elastic Net** |
|---|---|---|---|---|
| **Penalty** | none | L2 | L1 | L1 + L2 mix |
| **Weights** | all free | all shrunk | most exactly 0 | mix of both |
| **Feature selection** | no | no | yes | partly |
| **Needs standardizing** | no | yes | yes | yes |
| **Hyperparameters** | none | alpha | alpha | alpha, l1_ratio |

### Key Takeaways

1. **Age is predictable from gene expression.** Linear gene effects alone reach an
   out-of-fold R² around 0.78; regularized linear models reach **R² ≈ 0.9** on the
   held-out test set (MAE 5-6 years).
2. **Regularization is not optional here.** With 1,077 features and 652 samples, plain
   OLS over-fits; the penalized models reduce RMSE by roughly 3 years.
3. **LASSO selects genes.** About 180 of 1,077 genes get non-zero weights, with
   essentially no loss of accuracy.
4. **Hyperparameters are tuned by CV on the training data**; the test set is touched
   **once**, at the very end.
5. **Scale matters.** Penalized methods need standardized features, and the
   standardizer belongs *inside* the pipeline, re-fit per CV fold.

In [ ]:
# ============================================================
# EXERCISE
# ============================================================
#
# 1. The Ridge grid search tuned alpha against CV RMSE. Change
#    scoring='neg_root_mean_squared_error' to scoring='r2' and plot
#    mean_test_score directly (no minus sign). Does the best alpha change?
#
# 2. Extend the Elastic Net l1_ratio grid to [0.1, 0.5, 0.9, 0.95, 0.99].
#    Which mixture wins, and how many genes does the winning model use?
#    (Hint: np.count_nonzero on the refitted coefficients.)
#
# 3. Compare every model against the trivial baseline that always predicts
#    the mean training age (Hint: np.full_like(y_test, y_train.mean())).
#    How large is the improvement over this baseline?
#
# 4. Wrap Lasso(alpha=0.063) feature selection into the Pipeline so it is
#    re-fit inside every CV fold. Compare the selected gene count with
#    Part 4.

# Your code here:


## References

1. James G., Witten D., Hastie T., Tibshirani R. (2013). *An Introduction to Statistical Learning: with Applications in R*. Springer. Chapters 3 (linear regression), 5 (cross-validation), 6.2 (ridge and LASSO).
2. González-Velasco O., Papy-García D., Le Douaron G., Sánchez-Santos J. M., De Las Rivas J. (2020). Transcriptomic landscape, gene signatures and regulatory profile of aging in the human brain. *Biochimica et Biophysica Acta (BBA) - Gene Regulatory Mechanisms*, 1863(8), 194491. https://doi.org/10.1016/j.bbagrm.2020.194491 (source of the dataset and the 1,077-gene brain-ageing panel).
3. Friedman J., Hastie T., Tibshirani R. (2010). Regularization paths for generalized linear models via coordinate descent. *Journal of Statistical Software*, 33(1), 1-22 (the path algorithm behind `LassoCV` and `cv.glmnet`).
4. Pedregosa F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.